<a href="https://colab.research.google.com/github/Pranshu244/Urban-Flooding-Hydrology-Engine/blob/main/data_preparation/Hydrology_Engine_GIS_Data_Preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install rasterio

In [ ]:
!pip install earthengine-api geemap

In [ ]:
!rm -rf ~/.config/earthengine

In [ ]:
!pip install netCDF4

In [ ]:
import netCDF4 as nc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.plot import plotting_extent
from rasterio.transform import rowcol
from rasterio.features import rasterize
import geopandas as gpd
from PIL import Image
import geemap
from scipy.ndimage import uniform_filter
from scipy.spatial import cKDTree
ds=nc.Dataset('RF25_ind2025_rfp25.nc')

In [ ]:
file_path = 'RF25_ind2025_rfp25.nc'

In [ ]:
print(ds.variables.items())

In [ ]:
lats = ds.variables['LATITUDE'][:]
lons = ds.variables['LONGITUDE'][:]
time = ds.variables['TIME'][:]
rain = ds.variables['RAINFALL'][:]

In [ ]:
rain.shape

In [ ]:
lat_idx = np.where((lats >= 28.4) & (lats <= 28.9))[0]
lon_idx = np.where((lons >= 76.8) & (lons <= 77.35))[0]

In [ ]:
delhi_rain = rain[:, lat_idx[:, None], lon_idx]
delhi_rain.shape

In [ ]:
records = []
for t in range(len(time)):
    for i in lat_idx:
        for j in lon_idx:
            val = rain[t, i, j]
            if val != -999:
                records.append({
                    "day": t,
                    "lat": lats[i],
                    "lon": lons[j],
                    "rainfall_mm": val
                })

df = pd.DataFrame(records)
df.head()

In [ ]:
df.tail()

In [ ]:
df.shape

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
df.select_dtypes(include='number').columns

In [ ]:
for i in ['day', 'lat', 'lon']:
  sns.scatterplot(data=df,x=i,y='rainfall_mm')
  plt.show()

In [ ]:
import ee
ee.Authenticate(force=True)

In [ ]:
ee.Initialize(project='YOUR PROJECT NAME')
print("Earth Engine READY")

In [ ]:
admin = ee.FeatureCollection("FAO/GAUL/2015/level2")
delhi = admin.filter(
    ee.Filter.And(
        ee.Filter.eq('ADM0_NAME', 'India'),
        ee.Filter.eq('ADM1_NAME', 'Delhi')
    )
)
print(delhi.size().getInfo())

In [ ]:
dem = ee.Image("USGS/SRTMGL1_003")

In [ ]:
delhi_dem = dem.clip(delhi)

In [ ]:
stats = delhi_dem.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=delhi.geometry(),
    scale=30,
    maxPixels=1e9
)
print(stats.getInfo())

In [ ]:
task = ee.batch.Export.image.toDrive(
    image=delhi_dem,
    description='Delhi_DEM_30m',
    folder='GEE_Flood_Project',
    fileNamePrefix='delhi_dem_30m',
    region=delhi.geometry(),
    scale=30,
    maxPixels=1e13
)
task.start()
print("Export started")

In [ ]:
task.status()

In [ ]:
img = Image.open('delhi_dem_30m.tif')
img_array = np.array(img)

plt.figure(figsize=(8,8))
plt.imshow(img_array)
plt.axis('off')
plt.show()


In [ ]:
df.info()

In [ ]:
with rasterio.open("delhi_dem_30m.tif") as src:
    dem = src.read(1).astype(float)
    transform = src.transform
    dem_shape = dem.shape
dem[dem == 0] = np.nan

In [ ]:
rows, cols = dem.shape
xs, ys = np.meshgrid(np.arange(cols), np.arange(rows))
lons, lats = rasterio.transform.xy(transform, ys, xs)

In [ ]:
dem_df = pd.DataFrame({
    "lat": np.array(lats).ravel(),
    "lon": np.array(lons).ravel(),
    "elevation_m": dem.ravel()
})
dem_df = dem_df.dropna().reset_index(drop=True)

print("DEM DF shape:", dem_df.shape)

In [ ]:
dem_df.describe()

In [ ]:
df.describe()

In [ ]:
np.unique(dem[:1000])

In [ ]:
dem_df.shape

In [ ]:
rain_agg = df.groupby(["lat", "lon"])["rainfall_mm"].max().reset_index()
rain_agg.rename(columns={"rainfall_mm": "max_rainfall_mm"}, inplace=True)
print(rain_agg)

In [ ]:
rain_spatial = (
    df
    .groupby(["lat", "lon"])["rainfall_mm"]
    .quantile(0.95)
    .reset_index()
    .rename(columns={"rainfall_mm": "rain_95p_mm"})
)
print(rain_spatial)

In [ ]:
rain_coords = rain_spatial[["lat", "lon"]].values
tree = cKDTree(rain_coords)

dem_coords = dem_df[["lat", "lon"]].values
_, idx = tree.query(dem_coords, k=1)

dem_df["rain_95p_mm"] = rain_spatial.iloc[idx]["rain_95p_mm"].values

In [ ]:
dem_df[["elevation_m", "rain_95p_mm"]].describe()

In [ ]:
dem_df.isnull().sum()

In [ ]:
dem_df.head()

In [ ]:
for i in ['lat', 'lon', 'elevation_m']:
  sns.scatterplot(data=dem_df,x=i,y='rain_95p_mm')
  plt.show()

In [ ]:
dem_df.describe()

In [ ]:
dem_df.shape

In [ ]:
drains = gpd.read_file("export.geojson")
drains = drains[drains.geometry.notnull()]
drains = drains[drains.geometry.type == "LineString"]
print(drains.geometry.type.value_counts())

In [ ]:
drains = drains[drains.geometry.type == "LineString"]
print(len(drains))

In [ ]:
print(drains.crs)

In [ ]:
ax = drains.plot(figsize=(6,10), linewidth=0.6)
ax.set_aspect('equal')
plt.title("Delhi Drainage Network (OSM)")
plt.show()

In [ ]:
drain_raster = rasterize(
    [(geom, 1) for geom in drains.geometry],
    out_shape=dem_shape,
    transform=transform,
    fill=0,
    dtype=np.uint8
)

In [ ]:
print("Drain pixels:", drain_raster.sum())
print("Total pixels:", drain_raster.size)

In [ ]:
plt.figure(figsize=(6,10))
plt.imshow(drain_raster, cmap="Blues")
plt.title("Drainage Availability (Rasterized)")
plt.axis("off")
plt.show()

In [ ]:
drain_flat = drain_raster.ravel()
valid_mask = ~np.isnan(dem.ravel())
drain_flat_valid = drain_flat[valid_mask]
dem_df["drain_present"] = drain_flat_valid

In [ ]:
print(dem_df.shape)
print(dem_df.isna().sum())
print(dem_df.describe())
print(dem_df["drain_present"].value_counts())

In [ ]:
dem_df["drain_present"].value_counts()

In [ ]:
dem_df.describe()

In [ ]:
dem_df.sample(5)

In [ ]:
window_size = 9
drain_density = uniform_filter(
    drain_raster.astype(np.float32),
    size=window_size,
    mode="constant",
    cval=0.0
) * (window_size * window_size)

In [ ]:
print("Drain density stats:")
print("Min:", drain_density.min())
print("Max:", drain_density.max())
print("Mean:", drain_density.mean())
print("Median:", np.median(drain_density))

In [ ]:
plt.figure(figsize=(6,10))
plt.imshow(drain_density, cmap="Blues")
plt.title("Drain Density (9×9 neighborhood)")
plt.axis("off")
plt.show()

In [ ]:
drain_density_flat = drain_density.ravel()
valid_mask = ~np.isnan(dem.ravel())

In [ ]:
drain_density_valid = drain_density_flat[valid_mask]

In [ ]:
dem_df["drain_density"] = drain_density_valid

In [ ]:
dem_df[["drain_present", "drain_density"]].describe()

In [ ]:
dem_df["drain_density"].value_counts().head()

In [ ]:
dem_df.describe()

In [ ]:
final_cols = [
    "lat",
    "lon",
    "elevation_m",
    "rain_95p_mm",
    "drain_present",
    "drain_density"
]
hydrology_df = dem_df[final_cols].copy()
hydrology_df["drain_present"] = hydrology_df["drain_present"].astype("uint8")
output_path = "Hydrology_Engine_training.csv"
hydrology_df.to_csv(output_path, index=False)
print("✅ Dataset saved as:", output_path)
print("Shape:", hydrology_df.shape)
hydrology_df.head()

In [ ]:
delhi = ee.FeatureCollection("FAO/GAUL/2015/level2").filter(ee.Filter.eq('ADM2_NAME', 'Delhi'))
s1_collection = (ee.ImageCollection('COPERNICUS/S1_GRD')
                 .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
                 .filter(ee.Filter.eq('instrumentMode', 'IW'))
                 .select('VH')
                 .filterBounds(delhi))
before = s1_collection.filterDate('2023-07-01', '2023-07-07').mosaic().clip(delhi)
after = s1_collection.filterDate('2023-07-10', '2023-07-15').mosaic().clip(delhi)
flood_diff = after.subtract(before)
flood_mask = flood_diff.lt(-3).rename('flood_actual')
Map = geemap.Map()
Map.centerObject(delhi, 11)
Map.addLayer(before, {'min': -25, 'max': 0}, 'Pre-Flood (July 1)')
Map.addLayer(after, {'min': -25, 'max': 0}, 'Peak-Flood (July 13)')
Map.addLayer(flood_mask.selfMask(), {'palette': 'red'}, 'Sentinel-1 Actual Flood 2023')
Map

In [ ]:
task = ee.batch.Export.image.toDrive(
    image=flood_mask,
    description='Delhi_Actual_Flood_2023',
    scale=30,
    region=delhi.geometry(),
    fileFormat='GeoTIFF'
)
task.start()